# 학습 중 N/V 비중을 상보적으로 바꾸는 저차원 M2 — Dunnhumby seed 42

기존 저차원 M2 표현은 유지하고, 학습할 때만 거래활동(N)과 거래당 가치(V)의 상대 비중을 반대 방향으로 움직입니다. 두 축의 합계 개입량은 항상 같습니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 표현: ID 64차원 + 거래활동 4차원 + 거래당 가치 4차원
- 학습 점수: `S_ID + 0.1(1+ε)S_N + 0.1(1-ε)S_V`, `ε ~ Uniform(-0.3, 0.3)`
- 평가 점수: `S_ID + 0.1S_N + 0.1S_V`
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch
- 새로 학습: 이번 M2 한 개만. M1@64 seed 42는 동일 분할·설정·입력일 때만 기존 결과 재사용

이 실행은 역사적 개발구간의 seed 42 탐색이며 유의성을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '7919e2a1eff49426d6c9340bca9bb6f32982cbe6'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_gatefree_lowdim_balanced_training import (
    configure_balanced_training_run,
    preflight_summary,
    run_balanced_training_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_balanced_training_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_balanced_training_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['m2']['fixed_per_axis_budget'] == 0.10
assert summary['m2']['training_axis_balance_delta'] == 0.30
assert summary['m2']['training_total_axis_budget'] == 0.20
assert summary['m2']['evaluation_score_formula'] == 'S_ID + 0.1*S_N + 0.1*S_V'
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_balanced_training_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)